# 01 — WA Geophysical Data Exploration

**Goal:** Download public geophysical grids for the Yilgarn Craton / Eastern Goldfields (WA),
load known nickel-copper deposit locations from OZMIN, and visualise the data to build
intuition for the prospectivity mapping task.

**Region:** Eastern Goldfields, Yilgarn Craton, WA  
**Targets:** Komatiite-hosted Ni-Cu sulfide deposits (Glencore: Mt Keith, Cosmos, Murrin Murrin belt)  
**Data sources:**
- Geoscience Australia National Geophysical Grids (WCS)
- GSWA 80m TMI and gravity (higher res)
- OZMIN mineral deposits database

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from pathlib import Path

from geo_ml.ingest.ga_geophysics import YILGARN_BBOX, KALGOORLIE_BBOX, LAYERS
from geo_ml.ingest.ozmin import load_deposits, filter_wa_deposits, get_positive_labels

DATA_RAW = Path('../data/raw')
DATA_RAW.mkdir(exist_ok=True)

GDB_PATH = DATA_RAW / 'aus_mineral_deposits' / 'Mineral_Deposits_v01_20130729' / 'Mineral_Deposits.gdb'

print('Setup complete.')

## 1. Download Geophysical Grids

First run only — downloads from GA WCS. ~few minutes per layer.  
Skip if `data/raw/*.tif` files already exist.

In [ ]:
from geo_ml.ingest.ga_geophysics import download_grid

# Use Kalgoorlie-focused bbox for faster download during initial exploration
BBOX = KALGOORLIE_BBOX  # (120.5, -32.0, 122.5, -29.5)

layers_to_download = ['mag_tmi', 'gravity', 'rad_k', 'rad_th', 'rad_u']

for layer in layers_to_download:
    out_path = DATA_RAW / f'{layer}.tif'
    if out_path.exists():
        print(f'{layer}: already downloaded')
        continue
    print(f'Downloading {layer}...')
    try:
        download_grid(layer, bbox=BBOX, output_path=out_path)
        print(f'  -> {out_path}')
    except Exception as e:
        print(f'  Failed: {e}')

## 2. Load and Visualise Magnetic Data

In [ ]:
import rasterio
from rasterio.plot import show

mag_path = DATA_RAW / 'mag_tmi.tif'

with rasterio.open(mag_path) as src:
    mag = src.read(1).astype(np.float32)
    mag[mag == src.nodata] = np.nan
    transform = src.transform
    crs = src.crs

print(f'Shape: {mag.shape}')
print(f'Range: {np.nanmin(mag):.1f} to {np.nanmax(mag):.1f} nT')

fig, ax = plt.subplots(figsize=(10, 8))
vmin, vmax = np.nanpercentile(mag, [2, 98])
im = ax.imshow(mag, cmap='RdBu_r', vmin=vmin, vmax=vmax)
plt.colorbar(im, ax=ax, label='TMI (nT)')
ax.set_title('Total Magnetic Intensity — Eastern Goldfields, WA')
plt.tight_layout()
plt.show()

## 3. Compute Magnetic Derivatives

In [ ]:
from geo_ml.features.spatial import compute_magnetic_derivatives

derivs = compute_magnetic_derivatives(mag)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
titles = ['First Vertical Derivative', 'Analytic Signal', 'Tilt Angle']
keys = ['mag_fvd', 'mag_analytic_signal', 'mag_tilt_angle']
cmaps = ['seismic', 'inferno', 'coolwarm']

for ax, title, key, cmap in zip(axes, titles, keys, cmaps):
    arr = derivs[key]
    vmin, vmax = np.nanpercentile(arr, [2, 98])
    im = ax.imshow(arr, cmap=cmap, vmin=vmin, vmax=vmax)
    plt.colorbar(im, ax=ax)
    ax.set_title(title)

plt.suptitle('Magnetic derivative transforms — komatiite structural proxies')
plt.tight_layout()
plt.show()

## 4. Load Australian Mineral Deposits

Dataset: Australian Mineral Deposits (Geoscience Australia) — 18,719 records.  
Already downloaded to `data/raw/aus_mineral_deposits/`.

In [ ]:
gdf_all = load_deposits(GDB_PATH)
gdf = filter_wa_deposits(gdf_all, commodities=['nickel', 'copper'])
print(f'WA Ni/Cu deposits (deduplicated): {len(gdf)}')
print()
print(gdf[['NAME', 'COMMODID', 'COMMOD_NAME', 'OPERATING_STATUS']].head(10))

In [ ]:
# Overlay deposits on TMI map
fig, ax = plt.subplots(figsize=(10, 8))
vmin, vmax = np.nanpercentile(mag, [2, 98])
im = ax.imshow(mag, cmap='RdBu_r', vmin=vmin, vmax=vmax,
               extent=[BBOX[0], BBOX[2], BBOX[1], BBOX[3]], origin='upper')
plt.colorbar(im, ax=ax, label='TMI (nT)')

local = gdf.cx[BBOX[0]:BBOX[2], BBOX[1]:BBOX[3]]
ax.scatter(local.geometry.x, local.geometry.y,
           c='cyan', s=50, marker='^', zorder=5, label=f'Known deposits (n={len(local)})')
ax.legend()
ax.set_title('TMI + Known Ni-Cu Deposits — Kalgoorlie region')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
plt.tight_layout()
plt.show()

## 5. Next Steps

1. **`02_feature_engineering.ipynb`** — align all grids, compute derivatives, build feature matrix
2. **`03_pu_learning.ipynb`** — train bagging-PU XGBoost model on deposit locations
3. **`04_prospectivity_map.ipynb`** — predict and export full probability map as GeoTIFF
4. **`05_validation.ipynb`** — spatial cross-validation, AUC-ROC, SHAP importance

**Key geological question to validate with domain expert (Adrian Herbert, Glencore):**  
Do high-prospectivity zones predicted by TMI + gravity correlate with the known
komatiite belts NE of Kalgoorlie? Does the model pick up Mt Keith / Cosmos / Murrin Murrin?